# Baseline

In [ ]:
!pip install -q torch transformers sentencepiece

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda")  # send to GPU

# Test translation function
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    # Move inputs to GPU
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    translated = model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

In [ ]:
sample_es = "El paciente presenta fiebre alta y dolor de cabeza."
print(translate(sample_es))
# Expected: "The patient has a high fever and headache."

# Prepare for fine-tuning

In [ ]:
import random

# Real medical vocabulary
symptoms_es = ["fiebre", "dolor de cabeza", "náuseas", "mareos", "fatiga", "tos seca", "pérdida de apetito"]
symptoms_en = ["fever", "headache", "nausea", "dizziness", "fatigue", "dry cough", "loss of appetite"]
diseases_es = ["gripe", "diabetes", "hipertensión", "asma", "artritis", "migraña"]
diseases_en = ["flu", "diabetes", "hypertension", "asthma", "arthritis", "migraine"]
meds_es = ["paracetamol", "ibuprofeno", "omeprazol", "loratadina", "metformina"]
meds_en = ["paracetamol", "ibuprofen", "omeprazole", "loratadine", "metformin"]

templates = [
    ("El paciente presenta {sintoma} y ha sido diagnosticado con {enfermedad}.", 
     "The patient presents {sintoma_en} and has been diagnosed with {enfermedad_en}."),
    ("Se recomienda tomar {medicamento} dos veces al día.", 
     "It is recommended to take {medicamento_en} twice a day."),
    ("El tratamiento con {medicamento} es efectivo para la {enfermedad}.", 
     "Treatment with {medicamento_en} is effective for {enfermedad_en}."),
    ("El síntoma principal es {sintoma}, que puede estar asociado a {enfermedad}.", 
     "The main symptom is {sintoma_en}, which may be associated with {enfermedad_en}."),
    ("El estudio clínico muestra mejoría tras administrar {medicamento}.", 
     "The clinical study shows improvement after administering {medicamento_en}."),
]

pairs = []
for _ in range(5000):
    t_es, t_en = random.choice(templates)
    sintoma = random.choice(symptoms_es)
    sintoma_en = symptoms_en[symptoms_es.index(sintoma)]
    enfermedad = random.choice(diseases_es)
    enfermedad_en = diseases_en[diseases_es.index(enfermedad)]
    medicamento = random.choice(meds_es)
    medicamento_en = meds_en[meds_es.index(medicamento)]
    
    src = t_es.format(sintoma=sintoma, enfermedad=enfermedad, medicamento=medicamento)
    tgt = t_en.format(sintoma_en=sintoma_en, enfermedad_en=enfermedad_en, medicamento_en=medicamento_en)
    pairs.append((src, tgt))

random.shuffle(pairs)
pairs = pairs[:4000]
print(f"✅ Generated {len(pairs)} medical sentence pairs")

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "es": [p[0] for p in pairs],
    "en": [p[1] for p in pairs]
})

def preprocess(examples):
    # Tokenize source (Spanish)
    model_inputs = tokenizer(examples["es"], max_length=128, truncation=True)
    # Tokenize target (English) – we only need input_ids for labels
    labels = tokenizer(text_target=examples["en"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=["es", "en"])
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

## Trainer

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./medical_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    # NO tokenizer argument here
)

trainer.train()

In [ ]:
model.save_pretrained("./medical_model")
tokenizer.save_pretrained("./medical_model")

# QA

In [1]:
# First, pin transformers to a version that COMET supports
!pip install -q transformers==4.36.2

# Now install COMET (it will bring its own compatible sub-dependencies)
!pip install -q unbabel-comet

In [2]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint

print("Transformers version:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Transformers version: 4.36.2
CUDA available: True


In [3]:
from transformers import MarianMTModel, MarianTokenizer
import torch

# Load your fine-tuned model
model_path = "./medical_model"
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

# Test samples (you can replace with real ones later)
test_src = [
    "El paciente presenta fiebre alta y tos seca.",
    "Se recomienda administrar ibuprofeno cada 8 horas.",
    "El tratamiento con metformina es efectivo para la diabetes tipo 2."
]
# For reference-based COMET, we need reference translations; use our synthetic generator to create them
test_ref = [
    "The patient presents high fever and dry cough.",
    "It is recommended to administer ibuprofen every 8 hours.",
    "Treatment with metformin is effective for type 2 diabetes."
]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:197: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
from comet import download_model, load_from_checkpoint

# Download once
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

# Translate
def translate(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

translations = translate(test_src)

# Prepare COMET input
comet_data = [{"src": s, "mt": t, "ref": r} for s, t, r in zip(test_src, translations, test_ref)]
seg_scores = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
comet_scores = seg_scores.scores

# Show results
for i, (src, mt, score) in enumerate(zip(test_src, translations, comet_scores)):
    print(f"Source: {src}")
    print(f"MT: {mt}")
    print(f"COMET: {score:.3f}")
    print("---")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Encoder model frozen.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100

Source: El paciente presenta fiebre alta y tos seca.
MT: The patient presents high fever and dry cough.
COMET: 0.964
---
Source: Se recomienda administrar ibuprofeno cada 8 horas.
MT: It is recommended to administer ibuprofen every 8 hours.
COMET: 0.972
---
Source: El tratamiento con metformina es efectivo para la diabetes tipo 2.
MT: Treatment with metformin is effective for diabetes type 2.
COMET: 0.969
---


## BERTScore

In [6]:
!pip install -q bert-score

In [7]:
from bert_score import score

# Compute BERTScore (F1) for each translation against its reference
P, R, F1 = score(translations, test_ref, lang="en", model_type="microsoft/deberta-xlarge-mnli", verbose=True)
bertscores = F1.tolist()

# Show combined metrics
for i, (cs, bs) in enumerate(zip(comet_scores, bertscores)):
    print(f"Segment {i}: COMET={cs:.3f}, BERTScore={bs:.3f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.48 seconds, 6.24 sentences/sec
Segment 0: COMET=0.964, BERTScore=1.000
Segment 1: COMET=0.972, BERTScore=1.000
Segment 2: COMET=0.969, BERTScore=0.975


In [8]:
# Calibrated thresholds (adjust after human feedback)
COMET_AUTO = 0.88
COMET_REPAIR_MIN = 0.75
BERTSCORE_MIN = 0.85

def classify(comet, bert):
    if comet >= COMET_AUTO and bert >= BERTSCORE_MIN:
        return "auto-accept", "green"
    elif comet >= COMET_REPAIR_MIN:
        return "auto-repair", "orange"
    else:
        return "human-review", "red"

for i, (cs, bs) in enumerate(zip(comet_scores, bertscores)):
    action, color = classify(cs, bs)
    print(f"Seg {i}: {action.upper()} (COMET={cs:.2f}, BERT={bs:.2f}) → {color}")

Seg 0: AUTO-ACCEPT (COMET=0.96, BERT=1.00) → green
Seg 1: AUTO-ACCEPT (COMET=0.97, BERT=1.00) → green
Seg 2: AUTO-ACCEPT (COMET=0.97, BERT=0.98) → green


## test

In [13]:
# ANSI colours
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"

# Thresholds
COMET_AUTO   = 0.88
COMET_REPAIR = 0.75

def colour_and_action(comet):
    if comet >= COMET_AUTO:
        return GREEN, "auto-accept"
    elif comet >= COMET_REPAIR:
        return YELLOW, "auto-repair"
    else:
        return RED, "human-review"

for i, (src, mt, cs) in enumerate(zip(test_src, translations, comet_scores)):
    colour, action = colour_and_action(cs)
    
    print(f"{'='*60}")
    print(f"Source:      {src}")
    print(f"Translation: {colour}{mt}{RESET}")
    print(f"Metrics:     COMET = {cs:.3f} → {colour}{action}{RESET}")
    print()

Source:      El paciente presenta fiebre alta y tos seca.
Translation: The patient presents high fever and dry cough.
Metrics:     COMET = 0.964 → auto-accept

Source:      Se recomienda administrar ibuprofeno cada 8 horas.
Translation: It is recommended to administer ibuprofen every 8 hours.
Metrics:     COMET = 0.972 → auto-accept

Source:      El tratamiento con metformina es efectivo para la diabetes tipo 2.
Translation: Treatment with metformin is effective for diabetes type 2.
Metrics:     COMET = 0.969 → auto-accept



In [14]:
# --- Synthetic examples to show all confidence levels ---

# Correct references (the gold standard)
samples = [
    {
        "src": "El paciente presenta fiebre alta y tos seca.",
        "ref": "The patient presents high fever and dry cough.",
        # a good translation
        "mt": "The patient presents high fever and dry cough."
    },
    {
        "src": "Se recomienda administrar ibuprofeno cada 8 horas.",
        "ref": "It is recommended to administer ibuprofen every 8 hours.",
        # slightly wrong: missing "every 8 hours"
        "mt": "It is recommended to administer ibuprofen."
    },
    {
        "src": "El tratamiento con metformina es efectivo para la diabetes tipo 2.",
        "ref": "Treatment with metformin is effective for type 2 diabetes.",
        # heavily wrong: wrong drug, missing key info
        "mt": "Treatment with aspirin is not effective."
    },
    {
        "src": "El paciente muestra mejoría tras el cambio de medicación.",
        "ref": "The patient shows improvement after the medication change.",
        # moderate error: changed meaning
        "mt": "The patient shows worsening after the medication change."
    }
]

# Prepare data for COMET (reference‑based)
comet_input = [{"src": s["src"], "mt": s["mt"], "ref": s["ref"]} for s in samples]
scores = comet_model.predict(comet_input, batch_size=8, gpus=1 if torch.cuda.is_available() else 0).scores

# Display with colour coding (same thresholds and colouring function as before)
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"
COMET_AUTO   = 0.88
COMET_REPAIR = 0.75

def colour_and_action(comet):
    if comet >= COMET_AUTO:
        return GREEN, "auto-accept"
    elif comet >= COMET_REPAIR:
        return YELLOW, "auto-repair"
    else:
        return RED, "human-review"

for i, s in enumerate(samples):
    cs = scores[i]
    colour, action = colour_and_action(cs)
    print(f"{'='*60}")
    print(f"Source:      {s['src']}")
    print(f"Translation: {colour}{s['mt']}{RESET}")
    print(f"Metrics:     COMET = {cs:.3f} → {colour}{action}{RESET}")
    print()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 13.86it/s]


Source:      El paciente presenta fiebre alta y tos seca.
Translation: The patient presents high fever and dry cough.
Metrics:     COMET = 0.964 → auto-accept

Source:      Se recomienda administrar ibuprofeno cada 8 horas.
Translation: It is recommended to administer ibuprofen.
Metrics:     COMET = 0.776 → auto-repair

Source:      El tratamiento con metformina es efectivo para la diabetes tipo 2.
Translation: Treatment with aspirin is not effective.
Metrics:     COMET = 0.662 → human-review

Source:      El paciente muestra mejoría tras el cambio de medicación.
Translation: The patient shows worsening after the medication change.
Metrics:     COMET = 0.902 → auto-accept



In [15]:
from comet import download_model, load_from_checkpoint

# Reference‑free COMET – only needs source + translation
qe_model_path = download_model("Unbabel/wmt20-comet-qe-da")
qe_model = load_from_checkpoint(qe_model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/479 [00:00<?, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt20-comet-qe-da/snapshots/2e7ffc84fb67d99cf92506611766463bb9230cfb/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoder model frozen.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [17]:
import re
import torch

# ----- Your longer Spanish medical text -----
long_text = (
    "El paciente ingresó por dolor torácico agudo acompañado de disnea. "
    "Se realizó un electrocardiograma que mostró elevación del segmento ST en derivaciones inferiores. "
    "El diagnóstico presuntivo fue infarto agudo de miocardio. "
    "Se administró aspirina, clopidogrel y trombolíticos de inmediato. "
    "La evolución clínica fue favorable, sin complicaciones hemorrágicas. "
    "El paciente fue trasladado a la unidad coronaria para monitorización continua. "
    "Tras 48 horas, se inició rehabilitación cardiaca temprana. "
    "El ecocardiograma de control mostró fracción de eyección conservada. "
    "El paciente fue dado de alta con indicación de seguimiento cardiológico y tratamiento farmacológico."
)

# ----- Simple sentence splitter (works well for Spanish medical text) -----
sentences = re.split(r'(?<=[.!?])\s+', long_text.strip())
sentences = [s.strip() for s in sentences if s.strip()]

print(f"Found {len(sentences)} sentences.\n")

# ----- Translate all sentences with your fine‑tuned model -----
inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
translated_ids = model.generate(**inputs)
translations = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_ids]

# ----- Compute COMET QE scores (reference‑free) -----
qe_data = [{"src": s, "mt": t} for s, t in zip(sentences, translations)]
qe_output = qe_model.predict(qe_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
comet_scores = qe_output.scores

# ----- Thresholds & colours -----
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"

COMET_AUTO   = 0.55
COMET_REPAIR = 0.35

def colour_and_action(comet):
    if comet >= COMET_AUTO:
        return GREEN, "auto-accept"
    elif comet >= COMET_REPAIR:
        return YELLOW, "auto-repair"
    else:
        return RED, "human-review"

# ----- Display: source + coloured translation + metrics -----
for i, (src, mt, cs) in enumerate(zip(sentences, translations, comet_scores)):
    colour, action = colour_and_action(cs)
    print(f"{'='*70}")
    print(f"Source:      {src}")
    print(f"Translation: {colour}{mt}{RESET}")
    print(f"Metrics:     COMET = {cs:.3f} → {colour}{action}{RESET}")
    print()

Found 9 sentences.



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 12.53it/s]


Source:      El paciente ingresó por dolor torácico agudo acompañado de disnea.
Translation: The patient entered for acute chest pain accompanied by dyspnea.
Metrics:     COMET = 0.505 → auto-repair

Source:      Se realizó un electrocardiograma que mostró elevación del segmento ST en derivaciones inferiores.
Translation: An electrocardiogram was performed which showed elevation of the ST segment in inferior leads.
Metrics:     COMET = 0.288 → human-review

Source:      El diagnóstico presuntivo fue infarto agudo de miocardio.
Translation: The presumptive diagnosis was acute myocardial infarction.
Metrics:     COMET = 0.658 → auto-accept

Source:      Se administró aspirina, clopidogrel y trombolíticos de inmediato.
Translation: It was administered aspirin, clopidogrel and thrombolytics immediately.
Metrics:     COMET = 0.284 → human-review

Source:      La evolución clínica fue favorable, sin complicaciones hemorrágicas.
Translation: The clinical evolution was favorable, without hemor

In [18]:
# Combine sentences into one continuous source text
source_paragraph = " ".join(sentences)

# Build the coloured translation paragraph
coloured_sentences = []
for mt, cs in zip(translations, comet_scores):
    colour, _ = colour_and_action(cs)   # uses the same function and thresholds
    coloured_sentences.append(f"{colour}{mt}{RESET}")
translation_paragraph = " ".join(coloured_sentences)

# --- Display ---
print("=" * 70)
print("SOURCE TEXT:")
print(source_paragraph)
print("\nTRANSLATION (coloured by confidence):")
print(translation_paragraph)
print("=" * 70)

SOURCE TEXT:
El paciente ingresó por dolor torácico agudo acompañado de disnea. Se realizó un electrocardiograma que mostró elevación del segmento ST en derivaciones inferiores. El diagnóstico presuntivo fue infarto agudo de miocardio. Se administró aspirina, clopidogrel y trombolíticos de inmediato. La evolución clínica fue favorable, sin complicaciones hemorrágicas. El paciente fue trasladado a la unidad coronaria para monitorización continua. Tras 48 horas, se inició rehabilitación cardiaca temprana. El ecocardiograma de control mostró fracción de eyección conservada. El paciente fue dado de alta con indicación de seguimiento cardiológico y tratamiento farmacológico.

TRANSLATION (coloured by confidence):
The patient entered for acute chest pain accompanied by dyspnea. An electrocardiogram was performed which showed elevation of the ST segment in inferior leads. The presumptive diagnosis was acute myocardial infarction. It was administered aspirin, clopidogrel and thrombolytics imme